In [1]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict, List, Optional, Literal
import subprocess
from openai import OpenAI
import json
import os
import re
import requests
from bs4 import BeautifulSoup


class Question(TypedDict):
    id: str
    text: str
    choices: List[str]   # ["A) ...", "B) ...", ...]
    correct: str         # "A" ~ "E"
    year: int
    grade: int

class GaussState(TypedDict):
    grade: int
    problem_bank: List[Question]
    current_question: Optional[Question]
    answered_count: int
    correct_count: int
    score: int
    last_answer: Optional[str]

#### Tools


def load_problems_from_past_contests(grade: int) -> List[Question]:
    """
    TODO:
      - CEMC Gauss Past Contests PDF를 파싱해서 JSON/DB로 저장해두고 여기서 로드.
      - 지금은 간단한 예시 dummy 데이터.
    """
    dummy_questions: List[Question] = [
        {
            "id": "2024-G7-Q1",
            "text": "What is 2 + 3?",
            "choices": ["A) 4", "B) 5", "C) 6", "D) 7", "E) 8"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
        {
            "id": "2024-G7-Q2",
            "text": "What is 10 - 4?",
            "choices": ["A) 5", "B) 6", "C) 7", "D) 8", "E) 9"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
    ]
    return dummy_questions




In [2]:
def init_session(state: GaussState) -> GaussState:
    """
    - 학년(grade)을 이미 state에 넣어두었다고 가정,
      실제 구현에서는 여기서 사용자에게 7/8을 물어보는 로직 추가.
    - 점수 관련 필드를 초기화.
    """
    grade = state.get("grade", 7)  # 디폴트 7학년
    return {
        "grade": grade,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }

def load_past_contests(state: GaussState) -> GaussState:
    """
    - grade에 맞는 과거 Gauss 문제를 로딩하여 problem_bank에 저장.
    """
    grade = state["grade"]
    problems = load_problems_from_past_contests(grade)
    return {
        "problem_bank": problems
    }

def select_question(state: GaussState) -> GaussState:
    """
    - answered_count를 인덱스로 사용해서 다음 문제를 선택.
    - 더 이상 문제가 없으면 current_question을 None으로 둘 수 있음.
    """
    idx = state["answered_count"]
    problem_bank = state["problem_bank"]
    if idx < len(problem_bank):
        q = problem_bank[idx]
    else:
        q = None  # 문제 고갈 (실제 구현에서는 처리 로직 추가)
    return {
        "current_question": q
    }

def ask_question(state: GaussState) -> GaussState:
    q = state.get("current_question")
    if q is None:
        print("[알림] 현재 질문이 없습니다.")
        return {"last_answer": None}

    # 1) 문제 출력
    print("\n--- 문제 ---")
    print(f"({q['id']}) {q['text']}")
    for ch in q["choices"]:
        print("  " + ch)

    # 2) 사용자 입력 받기
    raw = input("정답을 선택하세요 (A/B/C/D/E): ").strip().upper()
    if raw not in ["A", "B", "C", "D", "E"]:
        print("잘못된 입력입니다. 기본값 A로 처리합니다.")
        raw = "A"

    # 3) 메타 상태 업데이트
    return {
        "last_answer": raw
    }

def check_answer(state: GaussState) -> GaussState:
    q = state.get("current_question")
    last = state.get("last_answer")
    if q is None or last is None:
        return {}
    correct = q["correct"]
    new_correct_count = state["correct_count"] + (1 if last == correct else 0)
    new_answered = state["answered_count"] + 1
    new_score = state["score"] + (10 if last == correct else 0)

    print("정답입니다!" if last == correct else f"오답입니다. 정답: {correct}")

    return {
      "answered_count": new_answered,
      "correct_count": new_correct_count,
      "score": new_score,
      "current_question": None,  # 다음에 select_question에서 신규로 설정
    }

def decide_continue(state: GaussState) -> Literal["continue", "stop"]:
    if state["answered_count"] >= len(state["problem_bank"]):
        return "stop"
    return "continue"

def end_session(state: GaussState) -> GaussState:
    print(f"\n최종 점수: {state['score']}점")
    
    # 필요시 다음 세션을 위해 초기화
    return {
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
        "current_question": None,
    }



In [3]:
graph_builder = StateGraph(GaussState)  # Python API 기준[web:36][web:43]

# 노드 추가
graph_builder.add_node("init_session", init_session)
graph_builder.add_node("load_past_contests", load_past_contests)
graph_builder.add_node("select_question", select_question)
graph_builder.add_node("ask_question", ask_question)
graph_builder.add_node("check_answer", check_answer)
graph_builder.add_node("end_session", end_session)

# 엣지 연결
graph_builder.add_edge(START, "init_session")
graph_builder.add_edge("init_session", "load_past_contests")
graph_builder.add_edge("load_past_contests", "select_question")
graph_builder.add_edge("select_question", "ask_question")
graph_builder.add_edge("ask_question", "check_answer")

# 조건부 엣지 (라우터)
graph_builder.add_conditional_edges(
    "check_answer",
    decide_continue,           # router 함수
    {
        "continue": "select_question",
        "stop": "end_session",
    },
)

graph_builder.add_edge("end_session", END)

graph = graph_builder.compile()


graph.invoke(
    {
        "grade": 7,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }
)


--- 문제 ---
(2024-G7-Q1) What is 2 + 3?
  A) 4
  B) 5
  C) 6
  D) 7
  E) 8
잘못된 입력입니다. 기본값 A로 처리합니다.
오답입니다. 정답: B

--- 문제 ---
(2024-G7-Q2) What is 10 - 4?
  A) 5
  B) 6
  C) 7
  D) 8
  E) 9
정답입니다!

최종 점수: 10점


{'grade': 7,
 'problem_bank': [{'id': '2024-G7-Q1',
   'text': 'What is 2 + 3?',
   'choices': ['A) 4', 'B) 5', 'C) 6', 'D) 7', 'E) 8'],
   'correct': 'B',
   'year': 2024,
   'grade': 7},
  {'id': '2024-G7-Q2',
   'text': 'What is 10 - 4?',
   'choices': ['A) 5', 'B) 6', 'C) 7', 'D) 8', 'E) 9'],
   'correct': 'B',
   'year': 2024,
   'grade': 7}],
 'current_question': None,
 'answered_count': 0,
 'correct_count': 0,
 'score': 0,
 'last_answer': None}